# Elo Ratings
Each team gets one strength rating that updates after every game. Beating a strong team moves it more than beating a weak one. The rating before tip-off becomes a feature, so there is no leakage.

In [12]:
import pandas as pd

In [13]:
games = pd.read_csv('../data/games_2021_26.csv')

In [14]:
home = games[games['MATCHUP'].str.contains('vs.')]
away = games[games['MATCHUP'].str.contains('@')]
matchups = pd.merge(home, away, on='GAME_ID', suffixes=('_HOME', '_AWAY'))

## Put the games in order
Convert the dates from text, then sort oldest to newest. Elo must see games in the order they were played.

In [15]:
matchups['GAME_DATE_HOME'] = pd.to_datetime(matchups['GAME_DATE_HOME'])
matchups = matchups.sort_values(['GAME_DATE_HOME', 'GAME_ID']).reset_index(drop=True)

In [ ]:
elo = {}
home_pre_rtg = []
away_pre_rtg = []
K = 20
prev_season = None

In [ ]:
for _, row in matchups.iterrows():

    # Check if new season started. Soft reset ratings.
    if row['SEASON_ID_HOME'] != prev_season:
        for team in elo:
            elo[team] = 0.75 * elo[team] + 0.25 * 1500  
        prev_season = row['SEASON_ID_HOME']

    home_elo = elo.get(row['TEAM_ID_HOME'], 1500)
    away_elo = elo.get(row['TEAM_ID_AWAY'], 1500)

    home_pre_rtg.append(home_elo)
    away_pre_rtg.append(away_elo)

    # See markdown above for formula breakdown
    expected_home = 1 / (1 + 10 ** ((away_elo - home_elo) / 400))